# Benin GDELT 2025
This notebook explores how Benin is perceived internationally and highlights signals about diplomacy, tourism, and investment using the cleaned GDELT dataset.

In [1]:
import csv
from collections import defaultdict
from statistics import mean

FILE_PATH = "copy_events_benin_2025_labeled.csv"

def load_rows(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def to_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

rows = load_rows(FILE_PATH)
print(f"Rows loaded: {len(rows)}")

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except Exception:
    HAS_MPL = False
    print("matplotlib not available; charts will be skipped.")

Rows loaded: 31508


## URL quality check
We flag suspect URLs (e.g., .ng or keywords like naira) and remove rows where the article text does not mention Benin.

In [9]:
import re
import html
import unicodedata
from urllib.parse import urlparse
from urllib.request import Request, urlopen

FLAGGED_PATH = "copy_events_benin_2025_labeled_url_flagged.csv"
FILTERED_PATH = "copy_events_benin_2025_labeled_url_filtered.csv"

# URL keywords flagged as likely Nigeria-related (user request)
SUSPECT_KEYWORDS = ["naira", "naija", "nigeria"]
SUSPECT_HOST_PARTS = ["ng.com"]
SUSPECT_TLD = ".ng"

# Article text signals (normalized to ASCII)
CITY_MARKERS = ["benin city", "city of benin", "benin-city", "edo state"]
REPUBLIC_MARKERS = ["republic of benin", "benin republic"]

def normalize_text(text):
    # Remove accents so "benin" matches "benin" and "benin"
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    return text.lower()

def is_suspect_url(url):
    if not url:
        return False
    u = url.lower()
    if any(k in u for k in SUSPECT_KEYWORDS):
        return True
    host = urlparse(u).netloc
    if not host:
        return False
    if host.endswith(SUSPECT_TLD) or (".ng." in host):
        return True
    if any(part in host for part in SUSPECT_HOST_PARTS):
        return True
    # Optional catch for ng in path (conservative)
    if "/ng/" in u:
        return True
    return False

def fetch_text(url, timeout=12, max_bytes=2_000_000):
    try:
        req = Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urlopen(req, timeout=timeout) as resp:
            content_type = resp.headers.get("Content-Type", "")
            if "text" not in content_type and "html" not in content_type:
                return None
            raw = resp.read(max_bytes).decode("utf-8", errors="ignore")
    except Exception:
        return None

    raw = re.sub(r"(?is)<script.*?>.*?</script>", " ", raw)
    raw = re.sub(r"(?is)<style.*?>.*?</style>", " ", raw)
    raw = re.sub(r"(?is)<[^>]+>", " ", raw)
    text = html.unescape(raw)
    text = re.sub(r"\s+", " ", text).strip()
    return normalize_text(text)

if not rows:
    raise ValueError("No rows loaded.")

url_cache = {}
benin_missing_urls = set()
benin_city_urls = set()
suspect_count = 0
checked_count = 0

for r in rows:
    url = (r.get("SOURCEURL") or "").strip()
    suspect = is_suspect_url(url)
    r["SuspectUrl"] = "True" if suspect else "False"

    if not suspect or not url:
        r["BeninMentioned"] = ""
        r["BeninCityMentioned"] = ""
        r["BeninRepublicMentioned"] = ""
        r["RemoveReason"] = ""
        continue

    suspect_count += 1
    if url not in url_cache:
        url_cache[url] = fetch_text(url)
        checked_count += 1

    text = url_cache[url]
    if text is None:
        r["BeninMentioned"] = "Unknown"
        r["BeninCityMentioned"] = "Unknown"
        r["BeninRepublicMentioned"] = "Unknown"
        r["RemoveReason"] = "Unknown"
        continue

    has_benin = "benin" in text
    has_city = any(m in text for m in CITY_MARKERS) or re.search(r"\bbenin\s+city\b", text)
    has_republic = any(m in text for m in REPUBLIC_MARKERS)

    r["BeninMentioned"] = "True" if has_benin else "False"
    r["BeninCityMentioned"] = "True" if has_city else "False"
    r["BeninRepublicMentioned"] = "True" if has_republic else "False"
    r["RemoveReason"] = ""

    # Rule 1: remove if Benin is not mentioned at all
    if not has_benin:
        benin_missing_urls.add(url)
        r["RemoveReason"] = "Benin not mentioned"
        continue

    # Rule 2: remove if Benin City is mentioned without Benin Republic
    if has_city and not has_republic:
        benin_city_urls.add(url)
        r["RemoveReason"] = "Benin City (Nigeria)"

filtered_rows = [
    r for r in rows
    if (r.get("SOURCEURL") or "").strip() not in benin_missing_urls
    and (r.get("SOURCEURL") or "").strip() not in benin_city_urls
 ]

fieldnames = list(rows[0].keys())
for col in ("SuspectUrl", "BeninMentioned", "BeninCityMentioned", "BeninRepublicMentioned", "RemoveReason"):
    if col not in fieldnames:
        fieldnames.append(col)

with open(FLAGGED_PATH, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

with open(FILTERED_PATH, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(filtered_rows)

print("URL quality check done")
print(f"- Suspect URLs flagged: {suspect_count}")
print(f"- Unique suspect URLs fetched: {checked_count}")
print(f"- URLs removed (Benin not mentioned): {len(benin_missing_urls)}")
print(f"- URLs removed (Benin City only): {len(benin_city_urls)}")
print(f"- Rows before: {len(rows)}")
print(f"- Rows after : {len(filtered_rows)}")
print(f"- Output (flagged): {FLAGGED_PATH}")
print(f"- Output (filtered): {FILTERED_PATH}")

URL quality check done
- Suspect URLs flagged: 13747
- Unique suspect URLs fetched: 3573
- URLs removed (Benin not mentioned): 27
- URLs removed (Benin City only): 1620
- Rows before: 31508
- Rows after : 25379
- Output (flagged): copy_events_benin_2025_labeled_url_flagged.csv
- Output (filtered): copy_events_benin_2025_labeled_url_filtered.csv


### Note méthodologique — manquement initial sur certains médias nigérians
Lors du **premier filtrage “Naija/Nigeria”**, on a surtout détecté les URLs suspectes via des heuristiques simples (ex. `naira/naija/nigeria`, TLD `.ng`, ou quelques patterns génériques).
**Limite constatée :** beaucoup de médias nigérians n’utilisent pas forcément un domaine en `.ng`, et certains noms de sites n’étaient pas inclus → une partie des URLs Nigéria a donc été classée à tort en `SuspectUrl=False`.

**Correction (dans la continuité, ciblée)**
- On repart du fichier déjà produit `copy_events_benin_2025_labeled_url_flagged.csv` (on **capitalise** sur le travail précédent).
- On applique `NIGERIAN_DOMAINS_PATTERN` (liste élargie de domaines/sources nigérianes).
- On ne re-scrape **que** le delta : les lignes `SuspectUrl=False` dont l’URL matche ce pattern (donc les **Nigeria manqués**).
- Pour ces URLs, on réutilise la même logique que plus haut via `fetch_text()` :
  - si l’article ne mentionne pas `benin` → `RemoveReason = "Benin not mentioned"` (à supprimer)
  - si l’article parle de `Benin City` sans marqueurs de la République du Bénin → `RemoveReason = "Benin City (Nigeria)"` (à supprimer)
- Sorties : `copy_events_benin_2025_labeled_url_flagged_v2.csv` et `copy_events_benin_2025_labeled_url_filtered_v2.csv`.

In [2]:
def fetch_text(url, timeout=12, max_bytes=2_000_000):
    try:
        req = Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urlopen(req, timeout=timeout) as resp:
            content_type = resp.headers.get("Content-Type", "")
            if "text" not in content_type and "html" not in content_type:
                return None
            raw = resp.read(max_bytes).decode("utf-8", errors="ignore")
    except Exception:
        return None

    raw = re.sub(r"(?is)<script.*?>.*?</script>", " ", raw)
    raw = re.sub(r"(?is)<style.*?>.*?</style>", " ", raw)
    raw = re.sub(r"(?is)<[^>]+>", " ", raw)
    text = html.unescape(raw)
    text = re.sub(r"\s+", " ", text).strip()
    return normalize_text(text)


In [7]:
# Re-filter ONLY the URLs that were previously SuspectUrl=False but are Nigeria-related (pattern fix)
import csv
import re
import html
import unicodedata
from urllib.parse import urlparse
from urllib.request import Request, urlopen

INPUT_FLAGGED = "copy_events_benin_2025_labeled_url_flagged.csv"
OUTPUT_FLAGGED_V2 = "copy_events_benin_2025_labeled_url_flagged_v2.csv"
OUTPUT_FILTERED_V2 = "copy_events_benin_2025_labeled_url_filtered_v2.csv"

# Expanded Nigeria domains/sources pattern (fix previous under-coverage)
NIGERIAN_DOMAINS_PATTERN = (
    r"\.ng$|punchng|nigerianobserver|thisdaylive|saharareporters|"
    r"premiumtimesng|thenationonlineng|nationalguideng|dailytrust|"
    r"naijanews|nigerianeye|withinnigeria|nigeriasun|vanguardngr|"
    r"channelstv|opinionnigeria|informationng|thenewsnigeria|"
    r"tribuneonlineng|theeagleonline|blueprint\.ng|thecable\.ng|"
    r"legit\.ng|naija247|pulse\.ng|thenigerianvoice|bellanaija|"
    r"nationalaccord|prompnewsonline|thesun\.ng|guardian\.ng|"
    r"leadership\.ng|quicknews|myjoyonline"
 )
NIGERIAN_DOMAINS_RE = re.compile(NIGERIAN_DOMAINS_PATTERN, re.IGNORECASE)

def load_rows_csv(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def is_nigeria_url(url: str) -> bool:
    if not url:
        return False
    u = url.lower()
    try:
        host = urlparse(u).netloc
    except Exception:
        host = ""
    # match host first, then fallback to the whole URL string (host+path+query)
    if host and NIGERIAN_DOMAINS_RE.search(host):
        return True
    return bool(NIGERIAN_DOMAINS_RE.search(u))

# Make this cell runnable even if earlier cells weren't executed in the current kernel
if "CITY_MARKERS" not in globals():
    CITY_MARKERS = ["benin city", "city of benin", "benin-city", "edo state"]
if "REPUBLIC_MARKERS" not in globals():
    REPUBLIC_MARKERS = ["republic of benin", "benin republic"]
if "normalize_text" not in globals():
    def normalize_text(text):
        text = unicodedata.normalize("NFKD", text)
        text = text.encode("ascii", "ignore").decode("ascii")
        return text.lower()
if "fetch_text" not in globals():
    def fetch_text(url, timeout=12, max_bytes=2_000_000):
        try:
            req = Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urlopen(req, timeout=timeout) as resp:
                content_type = resp.headers.get("Content-Type", "")
                if "text" not in content_type and "html" not in content_type:
                    return None
                raw = resp.read(max_bytes).decode("utf-8", errors="ignore")
        except Exception:
            return None

        raw = re.sub(r"(?is)<script.*?>.*?</script>", " ", raw)
        raw = re.sub(r"(?is)<style.*?>.*?</style>", " ", raw)
        raw = re.sub(r"(?is)<[^>]+>", " ", raw)
        text = html.unescape(raw)
        text = re.sub(r"\s+", " ", text).strip()
        return normalize_text(text)

rows_flagged = load_rows_csv(INPUT_FLAGGED)
print(f"Loaded flagged rows: {len(rows_flagged)}")

# Existing removals from the previous pass (keep them)
existing_removed_urls = set()
for r in rows_flagged:
    url = (r.get("SOURCEURL") or "").strip()
    if not url:
        continue
    if r.get("RemoveReason") in ("Benin not mentioned", "Benin City (Nigeria)"):
        existing_removed_urls.add(url)

# Target: only the previously non-suspect rows that are Nigeria URLs (per expanded pattern)
url_cache = {}
candidate_rows = 0
candidate_unique_urls = 0
updated_rows = 0
unknown_rows = 0

new_benin_missing_urls = set()
new_benin_city_urls = set()

for r in rows_flagged:
    url = (r.get("SOURCEURL") or "").strip()
    if not url:
        continue

    old_suspect = (r.get("SuspectUrl") or "").strip()
    if old_suspect != "False":
        continue

    if not is_nigeria_url(url):
        continue

    # This is exactly the missed bucket: SuspectUrl=False but Nigeria-related by pattern
    candidate_rows += 1
    r["SuspectUrl"] = "True"

    if url not in url_cache:
        url_cache[url] = fetch_text(url)
        candidate_unique_urls += 1

    text = url_cache[url]
    if text is None:
        r["BeninMentioned"] = "Unknown"
        r["BeninCityMentioned"] = "Unknown"
        r["BeninRepublicMentioned"] = "Unknown"
        r["RemoveReason"] = "Unknown"
        unknown_rows += 1
        continue

    has_benin = "benin" in text
    has_city = any(m in text for m in CITY_MARKERS) or re.search(r"\bbenin\s+city\b", text)
    has_republic = any(m in text for m in REPUBLIC_MARKERS)

    r["BeninMentioned"] = "True" if has_benin else "False"
    r["BeninCityMentioned"] = "True" if has_city else "False"
    r["BeninRepublicMentioned"] = "True" if has_republic else "False"
    r["RemoveReason"] = ""

    # Apply the same rules as earlier cells
    if not has_benin:
        r["RemoveReason"] = "Benin not mentioned"
        new_benin_missing_urls.add(url)
    elif has_city and not has_republic:
        r["RemoveReason"] = "Benin City (Nigeria)"
        new_benin_city_urls.add(url)

    updated_rows += 1

removed_urls = existing_removed_urls | new_benin_missing_urls | new_benin_city_urls
filtered_rows_v2 = [
    r for r in rows_flagged
    if (r.get("SOURCEURL") or "").strip() not in removed_urls
 ]

fieldnames = list(rows_flagged[0].keys())
for col in ("SuspectUrl", "BeninMentioned", "BeninCityMentioned", "BeninRepublicMentioned", "RemoveReason"):
    if col not in fieldnames:
        fieldnames.append(col)

with open(OUTPUT_FLAGGED_V2, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows_flagged)

with open(OUTPUT_FILTERED_V2, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(filtered_rows_v2)

print("Targeted re-filter (v2) done")
print(f"- Candidate rows checked (old SuspectUrl=False AND Nigeria pattern): {candidate_rows}")
print(f"- Unique candidate URLs fetched now: {candidate_unique_urls}")
print(f"- Candidate rows updated with verdicts: {updated_rows}")
print(f"- Candidate rows left Unknown (fetch failed / non-text): {unknown_rows}")
print(f"- NEW URLs removed (Benin not mentioned): {len(new_benin_missing_urls)}")
print(f"- NEW URLs removed (Benin City only): {len(new_benin_city_urls)}")
print(f"- Rows before: {len(rows_flagged)}")
print(f"- Rows after : {len(filtered_rows_v2)}")
print(f"- Output (flagged v2): {OUTPUT_FLAGGED_V2}")
print(f"- Output (filtered v2): {OUTPUT_FILTERED_V2}")

Loaded flagged rows: 31508
Targeted re-filter (v2) done
- Candidate rows checked (old SuspectUrl=False AND Nigeria pattern): 2963
- Unique candidate URLs fetched now: 792
- Candidate rows updated with verdicts: 2914
- Candidate rows left Unknown (fetch failed / non-text): 49
- NEW URLs removed (Benin not mentioned): 2
- NEW URLs removed (Benin City only): 398
- Rows before: 31508
- Rows after : 23967
- Output (flagged v2): copy_events_benin_2025_labeled_url_flagged_v2.csv
- Output (filtered v2): copy_events_benin_2025_labeled_url_filtered_v2.csv


## Traitement des valeurs manquantes <a id='7-missing'></a>

In [1]:
import pandas as pd

INPUT_PATH = "../data/GDELT_events_benin_2025_cleaned.csv"

df = pd.read_csv(INPUT_PATH)
df_clean = df.copy()


In [2]:
# ── Imputation 1 : Actor1/2Type1Code 
# Si le type est vide et le nom est un pays connu → 'STATE'
# Sinon → 'UNKNOWN' (entité locale non classifiée)

KNOWN_COUNTRIES = {
    'BENIN', 'NIGERIA', 'GHANA', 'TOGO', 'NIGER', 'FRANCE', 'USA',
    'SENEGAL', 'AFRICA', 'MALI', 'BURKINA', 'CAMEROON', 'COTE',
    'IVORY', 'CHAD', 'CHINESE', 'RUSSIA', 'GERMANY', 'EUROPEAN'
}

for col_type, col_name in [('Actor1Type1Code', 'Actor1Name'),
                             ('Actor2Type1Code', 'Actor2Name')]:
    df_clean[col_type + '_filled'] = df_clean[col_type].copy()
    mask_state = (
        df_clean[col_type].isna()
        & df_clean[col_name].str.upper().isin(KNOWN_COUNTRIES)
    )
    df_clean.loc[mask_state, col_type + '_filled'] = 'STATE'
    df_clean[col_type + '_filled'] = df_clean[col_type + '_filled'].fillna('UNKNOWN')

print("Actor1Type1Code_filled :")
print(df_clean['Actor1Type1Code_filled'].value_counts().head(8).to_string())
print()
print("Actor2Type1Code_filled :")
print(df_clean['Actor2Type1Code_filled'].value_counts().head(8).to_string())


Actor1Type1Code_filled :
Actor1Type1Code_filled
STATE      9399
UNKNOWN    5713
GOV        3326
MIL         790
IGO         600
COP         506
CVL         504
EDU         424

Actor2Type1Code_filled :
Actor2Type1Code_filled
UNKNOWN    9076
STATE      8539
GOV        2275
MIL         788
CVL         438
MED         405
EDU         358
IGO         298


In [3]:
# ── Imputation 2 : Actor2CountryCode pour acteurs institutionnels ────────
# Si Actor2 est une institution (GOVERNMENT, POLICE…) et que l'action
# se déroule au Bénin (FIPS: BN) → imputer 'BEN' (ISO-3)

INSTITUTIONAL_ACTORS = {
    'GOVERNMENT', 'PRESIDENT', 'MILITARY', 'GOVERNOR',
    'MINIST', 'POLICE', 'COURT', 'PARLIAMENT'
}

mask_gov_benin = (
    df_clean['Actor2CountryCode'].isna()
    & df_clean['Actor2Name'].isin(INSTITUTIONAL_ACTORS)
    & (df_clean['ActionGeo_CountryCode'] == 'BN')
)

df_clean.loc[mask_gov_benin, 'Actor2CountryCode'] = 'BEN'

print(f"Actor2CountryCode imputé à 'BEN' (institutions béninoises) : {mask_gov_benin.sum()} lignes")


Actor2CountryCode imputé à 'BEN' (institutions béninoises) : 1099 lignes


In [4]:
df_clean.to_csv(INPUT_PATH, index=False)
print("Imputation complete. Saved:", INPUT_PATH)

Imputation complete. Saved: ../data/GDELT_events_benin_2025_cleaned.csv
